<a href="https://colab.research.google.com/github/Fqrina/ai-archive/blob/main/Fish_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.linear_model import LinearRegression
print("Libraries imported successfully!")

Libraries imported successfully!


# STEP 1: LOAD DATA

In [ ]:
print("1. LOAD DATA")

train_df = pd.read_csv('train (4).csv')
test_df = pd.read_csv('test (4).csv')

print(f"Train data shape: {train_df.shape}")
print("\nTrain data head():")
print(train_df.head())

print(f"\nTest data shape: {test_df.shape}")
print("\nTest data head():")
print(test_df.head())

print("\nTarget distribution:")
print(train_df['catch_weight'].describe())

1. LOAD DATA
Train data shape: (160, 4)

Train data head():
   id  water_temperature  fish_density  catch_weight
0   0              12.25          2.96         14.58
1   1              12.10          3.38         15.08
2   2              12.95          3.32         15.63
3   3              12.80          3.74         16.12
4   4              13.65          3.68         16.67

Test data shape: (40, 3)

Test data head():
   id  water_temperature  fish_density
0   0              12.25          2.96
1   1              12.10          3.38
2   2              12.95          3.32
3   3              12.80          3.74
4   4              13.65          3.68

Target distribution:
count    160.000000
mean      17.973500
std        1.835308
min       14.580000
25%       16.685000
50%       17.980000
75%       19.265000
max       21.360000
Name: catch_weight, dtype: float64


# STEP 2: EXPLORATORY DATA ANALYSIS (BASIC)

In [ ]:
print("2. BASIC EDA")

print("\nData types:")
print(train_df.dtypes)

print("\nMissing values:")
print(train_df.isnull().sum())

print("\nFeature statistics:")
print(train_df[['water_temperature', 'fish_density']].describe())

2. BASIC EDA

Data types:
id                     int64
water_temperature    float64
fish_density         float64
catch_weight         float64
dtype: object

Missing values:
id                   0
water_temperature    0
fish_density         0
catch_weight         0
dtype: int64

Feature statistics:
       water_temperature  fish_density
count         160.000000    160.000000
mean           15.325000      3.710000
std             2.018312      1.058443
min            12.100000      1.880000
25%            13.612500      2.885000
50%            15.325000      3.710000
75%            17.037500      4.535000
max            18.550000      5.540000


# STEP 3: PREPARE DATA

In [ ]:
from sklearn.preprocessing import StandardScaler
X = train_df[['water_temperature', 'fish_density']]
y = train_df['catch_weight']
scaler = StandardScaler()
Xscaled = scaler.fit_transform(X)

In [ ]:

print("3. PREPARE DATA")



X_train, X_val, y_train, y_val = train_test_split(
    Xscaled,
    y,
    test_size=0.2,
    random_state=42,
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

3. PREPARE DATA
Training set: (128, 2)
Validation set: (32, 2)


# STEP 4: EXAMPLE MODEL

In [ ]:
print("4. BASELINE MODEL")

# Dummy regressor keeps the baseline simple and reproducible.
dummy_model = DummyRegressor(strategy='mean')
dummy_model.fit(X_train, y_train)

val_predictions = dummy_model.predict(X_val)
val_rmse = root_mean_squared_error(y_val, val_predictions)
val_mae = mean_absolute_error(y_val, val_predictions)
val_r2 = r2_score(y_val, val_predictions)

print(f"Validation RMSE: {val_rmse:.4f}")
print(f"Validation MAE: {val_mae:.4f}")
print(f"Validation R2: {val_r2:.4f}")

4. BASELINE MODEL
Validation RMSE: 1.6314
Validation MAE: 1.3528
Validation R2: -0.0015


In [ ]:
train_df.head()

,id,water_temperature,fish_density,catch_weight
0,0,12.25,2.96,14.58
1,1,12.10,3.38,15.08
2,2,12.95,3.32,15.63
3,3,12.80,3.74,16.12
4,4,13.65,3.68,16.67


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
rmse = root_mean_squared_error(model.predict(X_val),  y_val)

In [ ]:
rmse

0.0028181794924283933

In [ ]:
from sklearn.svm import SVR
svmodel = SVR()
svmodel.fit(X_train, y_train)
rmse = root_mean_squared_error(svmodel.predict(X_val), y_val)
rmse

0.07694312713236597

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
histmodel = HistGradientBoostingRegressor()
treemodel = RandomForestRegressor(
      n_estimators=300,
    max_depth=14,
    min_samples_split=4,
    random_state=42,
    n_jobs=-1
)
histmodel.fit(X_train, y_train)
rmse = root_mean_squared_error(histmodel.predict(X_val), y_val)
print(rmse)
treemodel.fit(X_train, y_train)
rmse2 = root_mean_squared_error(treemodel.predict(X_val), y_val)
print(rmse2)

0.24640341242539346
0.0036972662030693416


# STEP 5: GENERATE SUBMISSION

In [ ]:
print("5. FINAL SUBMISSION")

# Train on the full dataset, then predict the test split.
# svmodel.fit(X, y)
# test_predictions = dummy_model.predict(test_df[['water_temperature', 'fish_density']])
test_predictions = histmodel.predict(test_df[['water_temperature', 'fish_density']])
submission = pd.DataFrame({
    'id': test_df['id'],
    'catch_weight': test_predictions,
})

print("Submission preview:")
print(submission.head(10))
print(f"\nSubmission shape: {submission.shape}")
print("\nPrediction distribution:")
print(pd.Series(test_predictions).describe())

submission.to_csv('/submission5.csv', index=False)
print("\n✓ submission.csv berhasil dibuat!")

5. FINAL SUBMISSION
Submission preview:
   id  catch_weight
0   0     21.041567
1   1     21.041567
2   2     21.041567
3   3     21.041567
4   4     21.041567
5   5     21.041567
6   6     21.041567
7   7     21.041567
8   8     21.041567
9   9     21.041567

Submission shape: (40, 2)

Prediction distribution:
count    4.000000e+01
mean     2.104157e+01
std      1.439189e-14
min      2.104157e+01
25%      2.104157e+01
50%      2.104157e+01
75%      2.104157e+01
max      2.104157e+01
dtype: float64

✓ submission.csv berhasil dibuat!


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but HistGradientBoostingRegressor was fitted without feature names
  warnings.warn(
